In [1]:
import pandas as pd
import tqdm as tqdm

In [2]:
from huggingface_hub import upload_folder

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Huggingface_Api_Key")
from huggingface_hub import login

login(token=secret_value_0)

In [4]:
from huggingface_hub import HfApi
from huggingface_hub import HfApi

# api = HfApi()

# repo_id = "YOUR_HUGGINGFACE_USERNAME/kenyan-media-bias-qwen"

# api.create_repo(
#     repo_id=repo_id,
#     private=True,
#     exist_ok=True
# )

# print("Repository:", repo_id)

In [5]:
import torch

print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

GPU count: 2
GPU: Tesla T4


In [6]:
import torch

print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))

GPU count: 2
GPU: Tesla T4


In [7]:
!pip install -q transformers datasets accelerate sentencepiece evaluate rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00


In [8]:
import pandas as pd

df = pd.read_parquet('/kaggle/input/datasets/rikufootballcorner/political-verified/political_verified.parquet')

df.columns

Index(['url', 'website', 'article_text', 'political_classification',
       'bias_classification', 'bias_reason', 'judge_evidence_present',
       'judge_evidence_type', 'judge_evidence_quote', 'judge_verifier_reason',
       'judge_independent_label', 'judge_initial_label_supported',
       'judge_initial_reason_supported', 'judge_disagreement_reason',
       'judge_confidence', 'luna_label_norm', 'agree', 'quote_grounded'],
      dtype='object')

In [9]:
print(df.columns)
print(df.shape)

Index(['url', 'website', 'article_text', 'political_classification',
       'bias_classification', 'bias_reason', 'judge_evidence_present',
       'judge_evidence_type', 'judge_evidence_quote', 'judge_verifier_reason',
       'judge_independent_label', 'judge_initial_label_supported',
       'judge_initial_reason_supported', 'judge_disagreement_reason',
       'judge_confidence', 'luna_label_norm', 'agree', 'quote_grounded'],
      dtype='object')
(16367, 18)


In [10]:
df.drop_duplicates(subset=['url'],inplace=True)

In [11]:
df_agree=df[df['agree']==True]

In [12]:
df_agree['bias_classification'].value_counts()

bias_classification
not_biased    10473
biased         1163
Name: count, dtype: int64

In [13]:
biased_df = df_agree[
    df_agree["bias_classification"] == "biased"
]

not_biased_df = df_agree[
    df_agree["bias_classification"] == "not_biased"
].sample(
    n=len(biased_df) * 3,
    random_state=42
)

df_3to1 = pd.concat(
    [biased_df, not_biased_df],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [14]:
df_3to1.to_csv('final_sample1.csv', index=False)

In [15]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temporary
train_df, temp_df = train_test_split(
    df_3to1,
    test_size=0.20,
    random_state=42,
    stratify=df_3to1["bias_classification"]
)

# Second split: 10% validation, 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["bias_classification"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["bias_classification"].value_counts())

print("\nValidation distribution:")
print(val_df["bias_classification"].value_counts())

print("\nTest distribution:")
print(test_df["bias_classification"].value_counts())

Train: 3721
Validation: 465
Test: 466

Train distribution:
bias_classification
not_biased    2791
biased         930
Name: count, dtype: int64

Validation distribution:
bias_classification
not_biased    349
biased        116
Name: count, dtype: int64

Test distribution:
bias_classification
not_biased    349
biased        117
Name: count, dtype: int64


In [16]:
PROMPT_TEMPLATE = """
You are an expert annotator evaluating framing bias in Kenyan political news articles.

TASK
Determine whether the article contains textual evidence of biased framing that favors or disadvantages a political actor, political side, or political perspective.

Evaluate the article's presentation, not whether its political claims are true or whether you agree with them.

ASSESSMENT CRITERIA

Consider the following indicators:
1. Loaded language: emotionally charged or evaluative wording that portrays an actor favorably or unfavorably beyond straightforward reporting.
2. One-sided framing: presenting one political side's interpretation while excluding relevant competing perspectives.
3. Unequal treatment: describing, scrutinizing, or contextualizing comparable political actors differently.
4. Selective emphasis: emphasizing particular facts, allegations, or events in a way that creates a discernible political slant.
5. Perspective omission: excluding a relevant perspective when the omission materially affects the article's framing.

EVIDENCE RULES

- Base your decision exclusively on the supplied article.
- Identify specific words, phrases, descriptions, quotations, or presentation choices that support your decision.
- Distinguish the journalist's framing from claims or opinions attributed to politicians, witnesses, or other sources.
- Do not assume that an article is biased because of the politician, party, institution, topic, or news organization involved.
- Political criticism, allegations, corruption reports, government failures, and opposition claims are not automatically evidence of media bias.
- Do not assume that every article must give equal space to every political side.
- Do not invent quotations, omitted perspectives, or contextual facts.
- If there is insufficient textual evidence of biased framing, classify the article as not_biased.

CLASSIFICATION

Use exactly one label:
- biased: identifiable textual evidence indicates that the article's framing favors or disadvantages a political actor, side, or perspective.
- not_biased: the article is predominantly factual or balanced, or the available text does not provide sufficient evidence of biased framing.

OUTPUT FORMAT

Return exactly these two fields:

bias_classification: biased OR not_biased
bias_reason: A concise explanation grounded in specific evidence from the article. For biased articles, identify the evidence and explain its framing effect. For not_biased articles, explain the relevant features supporting the decision or state that sufficient evidence of biased framing was not found.

Do not add headings, scores, recommendations, or additional fields.
"""

def make_text(row):
    return f"""{PROMPT_TEMPLATE}

ARTICLE:
{row["article_text"]}

ANSWER:
bias_classification: {row["bias_classification"]}
bias_reason: {row["bias_reason"]}
"""

In [17]:
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [18]:
train_df = train_df.copy()
val_df = val_df.copy()

train_df["text"] = train_df.apply(make_text, axis=1)
val_df["text"] = val_df.apply(make_text, axis=1)
test_df["text"] = test_df.apply(make_text, axis=1)

In [19]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]],
    preserve_index=False
)

In [20]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [21]:
# Run only if needed:
# %pip install -q -U "bitsandbytes>=0.46.1" peft accelerate transformers datasets

import torch
import transformers
import peft
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle first.")

print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
PEFT: 0.20.0
bitsandbytes: 0.50.2
CUDA available: True
GPU: Tesla T4


In [22]:
import torch
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    AutoPeftModelForCausalLM,
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

MODEL_NAME = "Mriku/kenyan-media-bias-qwen"

repo_files = HfApi().list_repo_files(MODEL_NAME)
is_adapter = "adapter_config.json" in repo_files

print("Repository:", MODEL_NAME)
print("Contains existing LoRA adapter:", is_adapter)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if is_adapter:
    # Continue training the EXISTING adapter weights.
    model = AutoPeftModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        is_trainable=True,
        trust_remote_code=True,
    )

    print("Loaded existing adapter for continued training.")

else:
    # Start from the existing fine-tuned FULL model weights.
    # A new LoRA adapter will be attached to those weights.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )

    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ],
    )

    model = get_peft_model(model, lora_config)

    print("Loaded existing full weights with a new LoRA adapter.")

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model.print_trainable_parameters()

print("Model ready.")

Repository: Mriku/kenyan-media-bias-qwen
Contains existing LoRA adapter: True


tokenizer_config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 14.8MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loaded existing adapter for continued training.
trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
Model ready.


In [23]:
from datasets import Dataset

MAX_LENGTH = 2048

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

print("Training examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))
print("Example token count:", len(train_tokenized[0]["input_ids"]))

Map:   0%|          | 0/3721 [00:00<?, ? examples/s]

Map:   0%|          | 0/465 [00:00<?, ? examples/s]

Training examples: 3721
Validation examples: 465
Example token count: 1578


In [24]:
import inspect
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

OUTPUT_DIR = "/kaggle/working/kenyan-media-bias-qwen-continued"
ADDITIONAL_EPOCHS = 2

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

args = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=ADDITIONAL_EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=0,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
    data_seed=42,
)
# Handle different Transformers versions.
params = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in params:
    args["eval_strategy"] = "epoch"
else:
    args["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

# Compatibility with newer and older Transformers versions.
trainer_params = inspect.signature(Trainer.__init__).parameters

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Epochs:", ADDITIONAL_EPOCHS)
print("Learning rate:", training_args.learning_rate)
print("Output directory:", OUTPUT_DIR)

Epochs: 2
Learning rate: 1e-05
Output directory: /kaggle/working/kenyan-media-bias-qwen-continued


In [25]:
train_result = trainer.train()

print("\nContinued training complete.")
print("Training loss:", train_result.training_loss)
print("Training metrics:", train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,1.030711,0.974469
2,1.041150,0.972047



Continued training complete.
Training loss: 1.0487779813774665
Training metrics: {'train_runtime': 19496.0157, 'train_samples_per_second': 0.382, 'train_steps_per_second': 0.048, 'total_flos': 1.5416983035568128e+17, 'train_loss': 1.0487779813774665, 'epoch': 2.0}


In [26]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.save_state()

print("Saved continued model to:", OUTPUT_DIR)

import os

for name in os.listdir(OUTPUT_DIR):
    print(name)

Saved continued model to: /kaggle/working/kenyan-media-bias-qwen-continued
tokenizer_config.json
training_args.bin
trainer_state.json
tokenizer.json
chat_template.jinja
checkpoint-932
README.md
adapter_model.safetensors
adapter_config.json
checkpoint-466


In [27]:
import re
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

def make_inference_prompt(article):
    return f"""{PROMPT_TEMPLATE}

ARTICLE:
{article}

ANSWER:
"""

def parse_prediction(response):
    match = re.search(
        r"bias_classification\s*:\s*(not_biased|biased)\b",
        response,
        flags=re.IGNORECASE,
    )

    if not match:
        return "unparsed"

    return match.group(1).lower()

def generate_predictions(model, frame, batch_size=1):
    model.eval()
    predictions = []
    responses = []

    for start in tqdm(range(0, len(frame), batch_size)):
        batch = frame.iloc[start:start + batch_size]

        prompts = [
            make_inference_prompt(article)
            for article in batch["article_text"].tolist()
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1536,
        )

        # With device_map="auto", put inputs on the embedding device.
        input_device = model.get_input_embeddings().weight.device
        inputs = {k: v.to(input_device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=160,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Remove padded and unpadded prompt tokens before decoding.
        for i, output in enumerate(outputs):
            prompt_length = inputs["attention_mask"][i].sum().item()
            new_tokens = output[prompt_length:]

            response = tokenizer.decode(
                new_tokens,
                skip_special_tokens=True,
            ).strip()

            responses.append(response)
            predictions.append(parse_prediction(response))

    return predictions, responses

In [28]:
test_predictions, test_responses = generate_predictions(
    trainer.model,
    test_df,
)

y_true = test_df["bias_classification"].tolist()

valid_idx = [
    i for i, prediction in enumerate(test_predictions)
    if prediction in {"biased", "not_biased"}
]

print("Test examples:", len(test_df))
print("Parsed predictions:", len(valid_idx))
print("Unparsed predictions:", len(test_df) - len(valid_idx))

if valid_idx:
    y_valid = [y_true[i] for i in valid_idx]
    p_valid = [test_predictions[i] for i in valid_idx]

    print("\nClassification report:")
    print(classification_report(
        y_valid,
        p_valid,
        labels=["biased", "not_biased"],
        digits=4,
        zero_division=0,
    ))

    print("Confusion matrix (rows=true, columns=predicted):")
    print(confusion_matrix(
        y_valid,
        p_valid,
        labels=["biased", "not_biased"],
    ))

    print("Accuracy on parsed predictions:",
          round(accuracy_score(y_valid, p_valid), 4))

    print("Macro-F1 on parsed predictions:",
          round(precision_recall_fscore_support(
              y_valid,
              p_valid,
              average="macro",
              zero_division=0,
          )[2], 4))

  0%|          | 0/466 [00:00<?, ?it/s]

Test examples: 466
Parsed predictions: 411
Unparsed predictions: 55

Classification report:
              precision    recall  f1-score   support

      biased     0.8043    0.4458    0.5736        83
  not_biased     0.8740    0.9726    0.9206       328

    accuracy                         0.8662       411
   macro avg     0.8392    0.7092    0.7471       411
weighted avg     0.8599    0.8662    0.8506       411

Confusion matrix (rows=true, columns=predicted):
[[ 37  46]
 [  9 319]]
Accuracy on parsed predictions: 0.8662
Macro-F1 on parsed predictions: 0.7471


In [29]:
test_results = test_df[
    ["url", "article_text", "bias_classification", "bias_reason"]
].copy()

test_results["predicted_classification"] = test_predictions
test_results["generated_response"] = test_responses

test_results["correct"] = (
    test_results["predicted_classification"]
    == test_results["bias_classification"]
)

test_results.to_csv(
    "/kaggle/working/continued_model_test_predictions.csv",
    index=False,
)

test_results.to_parquet(
    "/kaggle/working/continued_model_test_predictions.parquet",
    index=False,
)

display(test_results.head(10))

,url,article_text,bias_classification,bias_reason,predicted_classification,generated_response,correct
3800,https://citizen.digital/article/court-orders-n...,The High Court has directed Nairobi County and...,not_biased,The article presents the court’s procedural di...,not_biased,bias_classification: not_biased\nbias_reason: ...,True
3305,https://www.the-star.co.ke/counties/nyanza/202...,Kisumu Governor Anyang Nyong’o/ Faith Matete\n...,not_biased,The article presents the county government's d...,not_biased,bias_classification: not_biased\nbias_reason: ...,True
4616,https://www.the-star.co.ke/counties/nyanza/202...,Property destroyed inside the Homa Bay Tourist...,not_biased,"The article uses predominantly factual, attrib...",not_biased,bias_classification: not_biased\nbias_reason: ...,True
666,https://citizen.digital/news/we-will-defend-or...,President William Ruto has vowed to stand for ...,not_biased,The article attributes the allegations rather ...,not_biased,bias_classification: not_biased\nbias_reason: ...,True
554,https://www.tv47.digital/matiangi-hits-back-at...,Former Interior Cabinet Secretary Dr Fred Mati...,not_biased,The article attributes the competing claims to...,not_biased,bias_classification: not_biased\nbias_reason: ...,True
139,https://capitalfm.africa/kanja-assigns-1000-po...,"NAIROBI, Kenya, Jul 14 – The National Police S...",not_biased,"The article uses predominantly factual, proced...",not_biased,bias_classification: not_biased\nbias_reason: ...,True
3273,https://www.the-star.co.ke/news/africa/2026-05...,IG Douglas Kanja and DCI boss Mohammed Amin be...,not_biased,"The article uses some dramatic language, such ...",unparsed,The government has repeatedly blamed the viole...,False
1521,https://www.capitalfm.co.ke/news/2025/10/iebc-...,"NAIROBI, Kenya Oct 24 – The Independent Electo...",not_biased,The article uses predominantly factual and att...,not_biased,bias_classification: not_biased\nbias_reason: ...,True
2543,https://www.the-star.co.ke/news/2026-07-02-pho...,Siaya Governor James Orengo with Embakasi East...,biased,The article frames the case primarily through ...,not_biased,bias_classification: not_biased\nbias_reason: ...,False
66,https://www.capitalfm.co.ke/news/2026/04/moham...,"NAIROBI, Kenya, Apr 7 — Nyali MP and Mombasa g...",not_biased,The article clearly attributes the allegations...,not_biased,bias_classification: not_biased\nbias_reason: ...,True


In [30]:
from huggingface_hub import HfApi

NEW_REPO = "Mriku/kenyan-media-bias-qwen-continued"

api = HfApi()

api.create_repo(
    repo_id=NEW_REPO,
    repo_type="model",
    private=True,
    exist_ok=True,
)

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=NEW_REPO,
    repo_type="model",
)

print("Uploaded:", NEW_REPO)

Uploaded: Mriku/kenyan-media-bias-qwen-continued
